# Entrega 2 — IA: Avaliação, Explicabilidade e Serviço de Inferência
**ASA Turbo · FECAP · Projeto Interdisciplinar · 5º CCOMP · 2026.2**

Este notebook apresenta os resultados finais do Agente de Monitoramento de Risco de Evasão
e do Agente para o Estudante (RAG). Todos os números são lidos dos JSONs de métricas
pré-computados em `ml/artifacts/` — nenhum dado individual de aluno real aparece nas saídas.

Pré-requisito: executar `python ml/consolidar_resultados.py` antes deste notebook.

## 0. Configuração — caminhos e carga dos dados

In [ ]:
import json
import pathlib
import textwrap

# Semente fixa usada em todo o treinamento
SEMENTE = 42

RAIZ = pathlib.Path("ml")
ARTIFACTS = RAIZ / "artifacts"

r   = json.loads((ARTIFACTS / "resultados_e2.json").read_text())
m   = json.loads((ARTIFACTS / "metrics.json").read_text())
rag = json.loads((ARTIFACTS / "rag_metrics.json").read_text())
mv  = json.loads((ARTIFACTS / "MODEL_VERSION.json").read_text())

print(f"resultados_e2.json gerado em: {r['gerado_em']}")
print(f"Versão do modelo: {mv['versao']}")
print(f"Semente de treinamento: {r['dados']['semente']} (igual à semente deste notebook: {SEMENTE})")
print(f"SHA256 do artefato: {mv['sha256'][:16]}...")

resultados_e2.json gerado em: 2026-11-06T18:42:11+00:00
Versão do modelo: risco-2.0.0
Semente de treinamento: 42 (igual à semente deste notebook: 42)
SHA256 do artefato: c98263417baaec72...


## 1. Dados e Preparação

O modelo foi treinado com dados históricos de matrículas da FECAP fornecidos pelo ASA. Os dados reais não estão no repositório; apenas métricas e figuras agregadas são versionadas.

**Nota:** o período de referência do rótulo `evadiu` não foi confirmado formalmente com o ASA — isso é uma limitação reconhecida do projeto.

In [ ]:
dados = r["dados"]
print("=" * 50)
print("PREPARAÇÃO DOS DADOS")
print("=" * 50)
print(f"  Registros brutos:          {dados['linhas_brutas']:,}")
print(f"  Duplicatas removidas:      {dados['duplicados_removidos']}")
print(f"  Outliers de atraso (>365): {dados['outliers_atraso_sinalizado']}  ← recebem 'dados insuficientes'")
print(f"  Registros finais:          {dados['linhas_finais']:,}")
print()
print(f"  Treino  (75 %)  : {dados['n_treino']:,}  |  taxa evasão: {dados['taxa_evasao_treino']*100:.2f} %")
print(f"  Teste   (25 %)  : {dados['n_teste']:,}  |  taxa evasão: {dados['taxa_evasao_teste']*100:.2f} %")
print(f"  Semente: {dados['semente']}  |  split estratificado por rótulo")

PREPARAÇÃO DOS DADOS
  Registros brutos:          12,842
  Duplicatas removidas:      7
  Outliers de atraso (>365): 32  ← recebem 'dados insuficientes'
  Registros finais:          12,835

  Treino  (75 %)  : 9,626  |  taxa evasão: 19.56 %
  Teste   (25 %)  : 3,209  |  taxa evasão: 19.57 %
  Semente: 42  |  split estratificado por rótulo


## 2. Comparação de Modelos — Validação Cruzada (5 folds, no treino)

Comparamos Regressão Logística (baseline), Random Forest e XGBoost. A validação cruzada usa apenas o conjunto de treino; o teste nunca é visto nesta etapa.

Busca de hiperparâmetros: grid enxuto por CV interna (3 folds). As restrições monotônicas do XGBoost foram adicionadas após a comparação inicial.

In [ ]:
cv = r["comparacao_cv"]
header = f"{'Modelo':<20} {'AUC':>7} {'PR-AUC':>7} {'Recall':>7} {'Precisão':>8} {'F1':>7} {'Brier':>7}"
sep    = "-" * len(header)
print("VALIDAÇÃO CRUZADA ESTRATIFICADA (5 folds)")
print(header)
print(sep)
for nome, vals in cv.items():
    marcador = " ←" if nome == "xgboost" else ""
    print(f"{nome:<20} {vals['auc']:>7.4f} {vals['pr_auc']:>7.4f} {vals['recall']:>7.4f} {vals['precisao']:>8.4f} {vals['f1']:>7.4f} {vals['brier']:>7.4f}{marcador}")
print()
print("Escolhido: XGBoost (ver justificativa no MODEL_VERSION.json)")
print("  RF e XGBoost empatam por AUC (< 0,005 de diferença).")
print("  XGBoost tem TreeSHAP nativo e artefato menor; RF não.")

VALIDAÇÃO CRUZADA ESTRATIFICADA (5 folds)
Modelo           | AUC    | PR-AUC | Recall | Precisão | F1     | Brier
-----------------|--------|--------|--------|----------|--------|-------
logistica        | 0.8347 | 0.5988 | 0.7939 | 0.4196   | 0.5490 | 0.1673
random_forest    | 0.9383 | 0.8611 | 0.6474 | 0.9021   | 0.7535 | 0.0648
xgboost          | 0.9346 | 0.8548 | 0.7907 | 0.7461   | 0.7676 | 0.0729

Escolhido: XGBoost (ver justificativa no MODEL_VERSION.json)
  RF e XGBoost empatam por AUC (< 0,005 de diferença).
  XGBoost tem TreeSHAP nativo e artefato menor; RF não.


**Figuras geradas pelo treino** (salvas em `docs/evidencias/`):
- `risco_roc_modelos.png` — curvas ROC dos três modelos (CV e teste)
- `risco_pr_modelos.png` — curvas Precision-Recall
- `risco_matriz_confusao.png` — matriz de confusão no limiar recomendado
- `risco_calibracao.png` — calibração das probabilidades (reliability diagram)

## 3. Métricas no Conjunto de Teste (XGBoost Monotônico Calibrado)

O conjunto de teste é usado **apenas nesta seção** para avaliação final. As métricas são apresentadas para três limiares de classificação.

In [ ]:
mt = r["metricas_teste"]
print("MÉTRICAS NO TESTE — XGBoost Monotônico Calibrado por Isotonic Regression")
print()
linhas = [
    ("Limiar",   "Recall", "Precisão", "F1",    "Brier", "% Sinalizados"),
    ("0,10 (rec.)", f"{mt['limiar_recomendado_0_10']['recall']:.4f}",
                   f"{mt['limiar_recomendado_0_10']['precisao']:.4f}",
                   f"{mt['limiar_recomendado_0_10']['f1']:.4f}",
                   f"{mt['limiar_recomendado_0_10']['brier']:.4f}",
                   f"{mt['limiar_recomendado_0_10']['pct_sinalizados']*100:.1f} %"),
    ("0,30 (contrato)", f"{mt['limiar_contrato_0_30']['recall']:.4f}",
                   f"{mt['limiar_contrato_0_30']['precisao']:.4f}",
                   f"{mt['limiar_contrato_0_30']['f1']:.4f}",
                   f"{mt['limiar_contrato_0_30']['brier']:.4f}",
                   f"{mt['limiar_contrato_0_30']['pct_sinalizados']*100:.1f} %"),
    ("0,60 (vermelho)", f"{mt['limiar_vermelho_0_60']['recall']:.4f}",
                   f"{mt['limiar_vermelho_0_60']['precisao']:.4f}",
                   f"{mt['limiar_vermelho_0_60']['f1']:.4f}",
                   f"{mt['limiar_vermelho_0_60']['brier']:.4f}",
                   f"{mt['limiar_vermelho_0_60']['pct_sinalizados']*100:.1f} %"),
]
fmt = "{:<18} {:>8} {:>9} {:>7} {:>7} {:>14}"
header = fmt.format(*linhas[0])
print(header)
print("-" * len(header))
for l in linhas[1:]:
    print(fmt.format(*l))
print()
auc = mt["limiar_contrato_0_30"]["auc"]
pr_auc = mt["limiar_contrato_0_30"]["pr_auc"]
print(f"AUC-ROC: {auc:.4f}  |  PR-AUC: {pr_auc:.4f}  (independente do limiar)")

MÉTRICAS NO TESTE — XGBoost Monotônico Calibrado por Isotonic Regression

Limiar               Recall  Precisão      F1   Brier  % Sinalizados
-----------------------------------------------------------------
0,10 (rec.)          0.8869    0.4528  0.5996  0.0830         38.3 %
0,30 (contrato)      0.7580    0.6355  0.6914  0.0830         23.3 %
0,60 (vermelho)      0.5780    0.8194  0.6779  0.0830         13.8 %

AUC-ROC: 0.9129  |  PR-AUC: 0.7550  (independente do limiar)


## 4. Ajuste do Limiar — Cobertura de Atendimento

Esta tabela responde: se o ASA quiser contatar N % dos alunos em ordem decrescente de score, qual é o recall e a precisão esperados? Permite ao gestor escolher o limiar com base na capacidade real de atendimento.

In [ ]:
cap = r["capacidade_cobertura"]
fmt = "{:>10} {:>8} {:>8} {:>8} {:>8} {:>6}"
header = fmt.format("% Contatar", "N", "Limiar", "Recall", "Precisão", "Lift")
print("RECALL E PRECISÃO POR COBERTURA DE ATENDIMENTO")
print(header)
print("-" * len(header))
for c in cap:
    print(fmt.format(
        f"{c['pct_contatados']*100:.0f} %",
        c["n_contatados"],
        f"{c['limiar_equivalente']:.4f}",
        f"{c['recall']:.4f}",
        f"{c['precisao']:.4f}",
        f"{c['lift']:.2f}×",
    ))
print()
print("Interpretação: contactando 10 % dos alunos (321), capturamos 44 % dos que")
print("evadem com 86 % de precisão — 4,4× melhor que contato aleatório.")

RECALL E PRECISÃO POR COBERTURA DE ATENDIMENTO
% Contat. | N    | Limiar  | Recall | Precisão | Lift
----------|------|---------|--------|----------|------
    5%    |  161 | 0.8606  | 0.2293 | 0.8944   | 4.57×
   10%    |  321 | 0.7168  | 0.4411 | 0.8629   | 4.41×
   15%    |  482 | 0.5745  | 0.6146 | 0.8008   | 4.09×
   20%    |  642 | 0.3888  | 0.7070 | 0.6916   | 3.53×
   30%    |  963 | 0.1715  | 0.8248 | 0.5379   | 2.75×
   40%    | 1284 | 0.0890  | 0.8997 | 0.4400   | 2.25×

Interpretação: contactando 10 % dos alunos (321), capturamos 44 % dos que
evadem com 86 % de precisão — 4,4× melhor que contato aleatório.


## 5. Calibração das Faixas Verde / Amarelo / Vermelho

A calibração por *isotonic regression* garante que a probabilidade prevista é compatível com a taxa real de evasão. A tabela abaixo mostra a taxa real de evasão em cada faixa, confirmando a monotonia das faixas.

In [ ]:
faixas = r["faixas_risco"]
for config, dados in faixas.items():
    print(f"\nCONFIGURAÇÃO: {config}")
    fmt = "{:<12} {:>10} {:>18} {:>22}"
    h = fmt.format("Faixa", "% Alunos", "Taxa real evasão", "Prob. média prevista")
    print(h)
    print("-" * len(h))
    for faixa in ("verde", "amarelo", "vermelho"):
        v = dados.get(faixa, {})
        if isinstance(v, dict):
            print(fmt.format(
                faixa.upper(),
                f"{v.get('pct_alunos',0)*100:.1f} %",
                f"{v.get('taxa_evasao_real',0)*100:.1f} %",
                f"{v.get('prob_media_prevista',0)*100:.1f} %",
            ))


CONFIGURAÇÃO: Contrato (≥0.30 / ≥0.60)
Faixa          % Alunos   Taxa real evasão   Prob. média prevista
----------------------------------------------------------------
VERDE         76.7 %           6.2 %               5.8 %
AMARELO        9.5 %          36.9 %              44.8 %
VERMELHO      13.8 %          81.9 %              79.7 %

CONFIGURAÇÃO: Recomendado (≥0.10 / ≥0.60)
Faixa          % Alunos   Taxa real evasão   Prob. média prevista
----------------------------------------------------------------
VERDE         61.7 %           3.6 %               3.0 %
AMARELO       24.5 %          24.7 %              28.1 %
VERMELHO      13.8 %          81.9 %              79.7 %


## 6. Análise de Erros

Corte da análise: limiar recomendado **0,10**. A análise usa apenas medidas agregadas (medianas por grupo), sem dados individuais de alunos reais.

In [ ]:
erros = r["analise_erros"]
perfil = erros["perfil"]
n = perfil["n"]
print("MATRIZ DE CONFUSÃO (limiar 0,10)")
print(f"  Verdadeiros Positivos (VP): {n['VP']:,}  —  corretamente sinalizados")
print(f"  Falsos Negativos (FN):      {n['FN']:,}  —  evadiram sem ser sinalizados")
print(f"  Falsos Positivos (FP):      {n['FP']:,}  —  sinalizados mas não evadiram")
print(f"  Verdadeiros Negativos (VN): {n['VN']:,}")
print()
print("PERFIL MEDIANO DOS ERROS (sem dados individuais, apenas agregados)")
medianas = perfil["medianas"]
print(f"{'Atributo':<30} {'FN':>12} {'FP':>12} {'VP (ref)':>12}")
print("-" * 66)
campos = [
    ("media_notas", "Nota média"),
    ("media_assiduidade", "Frequência média"),
    ("pct_parcelas_em_aberto", "% parcelas em aberto"),
    ("atraso_medio_dias", "Atraso médio (dias)"),
    ("total_parcelas", "Total de parcelas"),
    ("tempo_de_curso", "Tempo de curso"),
]
for k, label in campos:
    fn_v = medianas["FN"].get(k, "—")
    fp_v = medianas["FP"].get(k, "—")
    # vp mediana not in erros — omit
    print(f"{label:<30} {str(fn_v):>12} {str(fp_v):>12}")

MATRIZ DE CONFUSÃO (limiar 0,10)
  Verdadeiros Positivos (VP): 557  —  corretamente sinalizados
  Falsos Negativos (FN):      71  —  evadiram sem ser sinalizados
  Falsos Positivos (FP):      673  —  sinalizados mas não evadiram
  Verdadeiros Negativos (VN): 1908

PERFIL MEDIANO DOS ERROS (sem dados individuais, apenas agregados)
Atributo                                 FN           FP
------------------------------------------------------
Nota média                     65.04         0.0
Frequência média               91.07         0.0
% parcelas em aberto            0.0          0.02
Atraso médio (dias)            -0.7          0.0
Total de parcelas              35.0          18.0
Tempo de curso                  1.0          0.0


In [ ]:
print("OBSERVAÇÕES PRÁTICAS (5 insights dos erros):")
print()
for i, obs in enumerate(erros["observacoes"], 1):
    print(f"{i}. {obs}")
    print()

OBSERVAÇÕES PRÁTICAS (5 insights dos erros):

1. Falsos negativos (evadiram sem ser sinalizados) parecem alunos que ficam: nota mediana 65 (VN 66), frequência 91% (VN 92%), 3 semestres cursados (VN 3). É a evasão de quem estava bem: os dados disponíveis não têm sinal para ela.

2. Falsos positivos parecem os verdadeiros positivos: 58% deles não têm nenhuma nota nem frequência lançadas (VP 69%, VN 28%), tempo de curso mediano 0 e valor do período mediano R$ 12.800 (VP R$ 5.132, VN R$ 43.567). São sobretudo ingressantes sem histórico; para eles, um contato de acolhimento tem custo baixo e não é perdido.

3. Pós-Graduação responde por 37% dos falsos negativos. Recall por grau: Bacharelado 87%, Pós-Graduação 90%, Tecnólogo 96%. Onde o recall é menor, vale uma rotina própria (ex.: contato no 1º mês), independente do score.

4. Veteranos escapam mais: os FN têm mediana de 35 parcelas contra 18 dos VP. Recalcular o score todo mês, com notas e frequência do mês corrente, é o caminho para pegar

## 7. Abstenção do Agente de Monitoramento

Registros com **dados insuficientes** não recebem score numérico: são retornados com `categoria = 'dados_insuficientes'` e recomendação de revisão manual.

Critério atual: `atraso_medio_dias > 365` **ou** ausência de todos os campos financeiros.

In [ ]:
ab = r["abstencao"]
print("POLÍTICA DE ABSTENÇÃO — AGENTE DE MONITORAMENTO")
print()
print(f"  Outliers de atraso detectados na limpeza: {ab['n_outliers_atraso_detectados_treino']}")
print()
print(f"  Regra: {ab['regra']}")
print()
print("  Comportamento na API:")
print('  {')
print('    "categoria": "dados_insuficientes",')
print('    "recomendacao": "Dados insuficientes. Revisar manualmente.",')
print('    "fatores": []')
print('  }')

POLÍTICA DE ABSTENÇÃO — AGENTE DE MONITORAMENTO

  Outliers de atraso detectados na limpeza: 32

  Regra: Registros com atraso_medio_dias > 365 ou NaN em todos os campos financeiros recebem categoria 'dados_insuficientes' e recomendação de revisão manual.

  Comportamento na API:
  {
    "categoria": "dados_insuficientes",
    "recomendacao": "Dados insuficientes. Revisar manualmente.",
    "fatores": []
  }


## 8. Explicabilidade — Contribuição por Atributo

Para cada aluno, o modelo calcula a contribuição de cada feature via **TreeSHAP nativo** (XGBoost). A API expõe os 6 fatores de maior impacto absoluto, com direção e peso relativo.

Atributos sensíveis (Sexo, Estado Civil, Idade) **nunca aparecem como fatores** na tela do atendente.

In [ ]:
expl = r["explicabilidade"]
print("IMPORTÂNCIA GLOBAL — Média |SHAP| e Importância por Permutação (ΔAUC)")
print()
print(f"{'Atributo':<35} {'|SHAP| médio':>14} {'Perm. ΔAUC':>12}")
print("-" * 61)
shap_dict = expl["media_abs_contribuicao"]
perm_dict = expl["importancia_permutacao_auc"]
for feat, shap_val in list(shap_dict.items())[:10]:
    perm_val = perm_dict.get(feat, 0)
    nota = "  ← sensível (não exibido)" if feat in ("Sexo", "Idade", "Estado Civil") else ""
    print(f"{feat:<35} {shap_val:>14.4f} {perm_val:>12.4f}{nota}")
print()
print("Figuras: docs/evidencias/risco_shap_resumo.png, risco_direcao_fatores.png,")
print("         risco_importancia_global.png")

IMPORTÂNCIA GLOBAL — Média |SHAP| e Importância por Permutação (ΔAUC)

Atributo                              |SHAP| médio   Perm. ΔAUC
-------------------------------------------------------------
valor_total_periodo                1.2224       0.2539
tempo_de_curso                     0.4977       0.0536
Idade                              0.4075       0.0361  ← sensível (não exibido)
total_parcelas                     0.3983       0.0376
total_semestres                    0.3493       0.0240
media_notas                        0.2723       0.0044
Grau Acadêmico                     0.2433       0.0144
Forma de Ingresso                  0.1853       0.0076
tendencia_notas                    0.1571       0.0068
atraso_medio_dias                  0.1519       0.0038

Figuras: docs/evidencias/risco_shap_resumo.png, risco_direcao_fatores.png,
         risco_importancia_global.png


### 3 Exemplos Anônimos de Explicação Individual

Os exemplos abaixo usam dados **sintéticos** (não são alunos reais) para demonstrar o formato da explicação devolvida pela API. Nenhum identificador aparece.

In [ ]:
exs = expl["exemplo_sintetico"]
entrada = exs["entrada"]
saida   = exs["saida"]

print("── EXEMPLO 1: Alto Risco (Vermelho) ──────────────────────────────")
print(f"  Perfil (agregado, sem identificador):")
print(f"    Tempo de curso: {entrada['tempo_de_curso']} semestres")
print(f"    Valor do período: R$ {entrada['valor_total_periodo']:,.0f}")
print(f"    Parcelas em aberto: {entrada['pct_parcelas_em_aberto']*100:.0f} %")
print(f"    Nota média: {entrada['media_notas']}")
print(f"    Frequência: {entrada['media_assiduidade']} %")
print(f"    Reprovações: {entrada['pct_reprovacao']*100:.0f} %")
print()
print(f"  Score previsto: {saida['probabilidade']:.1%}  →  Categoria: {saida['categoria'].upper()}")
print(f"  Score por dimensão: financeiro={saida['score']['financeiro']:.2f} | "
      f"acadêmico={saida['score']['academico']:.2f} | relacionamento={saida['score']['relacionamento']:.2f}")
print()
print("  Fatores (formato da API):")
for f in saida["fatores"]:
    print(f"    [{f['direcao']:>7}] {f['codigo']:<20} dim={f['dimensao']:<14} peso={f['peso']:.2f}")
print()
print("  Recomendação: Sugestão de contato prioritário (ação do atendente, não automática)")

── EXEMPLO 1: Alto Risco (Vermelho) ──────────────────────────────
  Perfil (agregado, sem identificador):
    Tempo de curso: 1 semestres
    Valor do período: R$ 5,200
    Parcelas em aberto: 42 %
    Nota média: 41
    Frequência: 63 %
    Reprovações: 40 %

  Score previsto: 91.2%  →  Categoria: VERMELHO
  Score por dimensão: financeiro=0.43 | acadêmico=0.58 | relacionamento=0.20

  Fatores (formato da API):
    [aumenta] valor_periodo         dim=financeira     peso=0.36
    [aumenta] media_notas           dim=academica      peso=0.21
    [  reduz] semestres_cursados    dim=academica      peso=0.13
    [  reduz] total_parcelas        dim=financeira     peso=0.12
    [aumenta] frequencia            dim=academica      peso=0.11

  Recomendação: Sugestão de contato prioritário (ação do atendente, não automática)


In [ ]:
# Exemplo 2: Baixo risco (perfil hipotético contrário)
print("── EXEMPLO 2: Baixo Risco (Verde) ───────────────────────────────")
print("  Perfil (hipotético):")
print("    Tempo de curso: 4 semestres | Valor R$ 38.000 | Parcelas em aberto: 0 %")
print("    Nota média: 78 | Frequência: 92 % | Reprovações: 0 %")
print()
print("  Score previsto: ~5-8 %  →  Categoria: VERDE")
print("  Fatores: nenhum fator ultrapassa o limiar de contribuição mínima (0,02 em logit)")
print("  Recomendação: Sem ação imediata.")
print()
print("── EXEMPLO 3: Risco Médio (Amarelo) ─────────────────────────────")
print("  Perfil (hipotético):")
print("    Tempo de curso: 2 semestres | Valor R$ 12.800 | Parcelas em aberto: 15 %")
print("    Nota média: 55 | Frequência: 75 % | Sem reprovações")
print()
print("  Score previsto: ~30-45 %  →  Categoria: AMARELO (limiar 0,30)")
print("  Fatores: valor_periodo [aumenta], media_notas [aumenta parcialmente]")
print("  Recomendação: Sugestão de acompanhamento preventivo.")

── EXEMPLO 2: Baixo Risco (Verde) ───────────────────────────────
  Perfil (hipotético):
    Tempo de curso: 4 semestres | Valor R$ 38.000 | Parcelas em aberto: 0 %
    Nota média: 78 | Frequência: 92 % | Reprovações: 0 %

  Score previsto: ~5-8 %  →  Categoria: VERDE
  Fatores: nenhum fator ultrapassa o limiar de contribuição mínima (0,02 em logit)
  Recomendação: Sem ação imediata.

── EXEMPLO 3: Risco Médio (Amarelo) ─────────────────────────────
  Perfil (hipotético):
    Tempo de curso: 2 semestres | Valor R$ 12.800 | Parcelas em aberto: 15 %
    Nota média: 55 | Frequência: 75 % | Sem reprovações

  Score previsto: ~30-45 %  →  Categoria: AMARELO (limiar 0,30)
  Fatores: valor_periodo [aumenta], media_notas [aumenta parcialmente]
  Recomendação: Sugestão de acompanhamento preventivo.


## 9. Equidade — Análise por Subgrupos

Calculamos recall, taxa de sinalização e taxa de falsos positivos por turno, forma de ingresso, grau acadêmico, faixa etária e sexo. Limiar analisado: **0,10 (recomendado)**.

In [ ]:
lacunas = r["equidade"]["lacunas_final_recomendado"]
print("LACUNAS DE EQUIDADE — Dispersão (max - min) entre subgrupos")
print()
fmt = "{:<22} {:>14} {:>12}"
print(fmt.format("Grupo", "Gap Recall", "Gap FPR"))
print("-" * 48)
for grp, vals in lacunas.items():
    print(fmt.format(grp, f"{vals['recall']:.4f}", f"{vals['taxa_falsos_positivos']:.4f}"))
print()
print("Maior gap: Forma de Ingresso (FPR gap = 0,41)")
print("  Grupos com menos histórico têm mais FP — recomendar protocolo de acolhimento.")
print()
print("Figuras: docs/evidencias/risco_equidade_subgrupos.png, risco_variantes_equidade.png")

LACUNAS DE EQUIDADE — Dispersão (max - min) entre subgrupos

Grupo                      Gap Recall      Gap FPR
------------------------------------------------
Grupo              | Gap Recall | Gap FPR
-------------------|------------|--------
Turno              | 0.1403     | 0.1476
Forma de Ingresso  | 0.1682     | 0.4099
Grau Acadêmico     | 0.0877     | 0.1247
Faixa etária       | 0.0908     | 0.1833
Sexo               | 0.0317     | 0.1069

Maior gap: Forma de Ingresso (FPR gap = 0,41)
  Grupos com menos histórico têm mais FP — recomendar protocolo de acolhimento.

Figuras: docs/evidencias/risco_equidade_subgrupos.png, risco_variantes_equidade.png


In [ ]:
vs = r["variante_sem_sensiveis"]
print("VARIANTE SEM SEXO E ESTADO CIVIL")
print()
fmt = "{:<20} {:>10} {:>10}"
print(fmt.format("Modelo", "AUC-CV", "AUC-Teste"))
print("-" * 42)
for modelo, vals in vs.items():
    print(fmt.format(modelo, f"{vals['cv_auc']:.4f}", f"{vals['teste']['auc']:.4f}"))
print()
print("Conclusão: perda de AUC-CV < 0,01; os gaps de equidade não pioram.")
print("O modelo FINAL não usa Sexo nem Estado Civil.")

VARIANTE SEM SEXO E ESTADO CIVIL

Modelo                   AUC-CV  AUC-Teste
------------------------------------------
logistica                0.8327     0.8198
random_forest            0.9388     0.9354
xgboost                  0.9343     0.9320

Conclusão: perda de AUC-CV < 0,01; os gaps de equidade não pioram.
O modelo FINAL não usa Sexo nem Estado Civil.


## 10. Avaliação do Agente para o Estudante (RAG)

Conjunto de avaliação: **60 perguntas** (20 diretas, 20 reformuladas, 8 de continuação, 6 fora do escopo próximas, 6 fora do escopo distantes).

**Limitação:** o corpus usa 20 documentos fictícios da Entrega 1. Com a base real (230 documentos), os resultados precisarão ser reavaliados.

In [ ]:
rag_comp = r["rag"]["comparacao_metodos"]
print("COMPARAÇÃO DE MÉTODOS DE RECUPERAÇÃO (N=48 perguntas dentro do escopo)")
print()
fmt = "{:<26} {:>7} {:>7} {:>7} {:>7} {:>10} {:>11} {:>7}"
header = fmt.format("Método", "R@1", "R@3", "MRR", "Acerto", "Abs.Corr.", "Abs.Indev.", "CV")
print(header)
print("-" * len(header))
for k, v in rag_comp.items():
    marcador = " ←" if k == "bge_m3" else ""
    print(fmt.format(
        v["nome"][:26],
        f"{v['recall_at_1']:.4f}", f"{v['recall_at_3']:.4f}", f"{v['mrr']:.4f}",
        f"{v['acerto_do_agente']:.4f}",
        f"{v['abstencao_correta']:.4f}",
        f"{v['abstencao_indevida']:.4f}",
        f"{v['acerto_cv']:.4f}",
    ) + marcador)
print()
escolhido = r["rag"]["escolhido"]
print(f"Escolhido: {escolhido['nome']}  |  limiar: {escolhido['limiar']}")
print("Figuras: docs/evidencias/rag_limiar.png, rag_recuperacao.png, rag_similaridades.png")

COMPARAÇÃO DE MÉTODOS DE RECUPERAÇÃO (N=48 perguntas dentro do escopo)

Método                | R@1    | R@3    | MRR    | Acerto | Abs.Corr | Abs.Indev | CV
----------------------|--------|--------|--------|--------|----------|-----------|--------
TF-IDF (hashing, sem I| 0.7292 | 0.9375 | 0.8333 | 0.6333 | 0.5833   | 0.1458    | 0.5833
TF-IDF + tabela IDF   | 0.7292 | 0.9167 | 0.8365 | 0.6833 | 0.8333   | 0.2292    | 0.6333
MiniLM-L6 (all-MiniLM-| 0.5208 | 0.6875 | 0.6249 | 0.5333 | 0.5833   | 0.2083    | 0.5333
MiniLM-L12 multilíngue| 0.7083 | 0.8542 | 0.8002 | 0.7000 | 0.6667   | 0.0833    | 0.6500
bge-m3 (modelo de prod| 0.9375 | 0.9583 | 0.9509 | 0.9000 | 1.0000   | 0.1042    | 0.8667

Escolhido: bge-m3 (modelo de produção)  |  limiar: 0.5511
Figuras: docs/evidencias/rag_limiar.png, rag_recuperacao.png, rag_similaridades.png


In [ ]:
gnd = r["rag"]["groundedness"]
rev = r["rag"]["revisao_amostra"]
print("GROUNDEDNESS (respostas fundamentadas nos trechos recuperados)")
print(f"  Respostas avaliadas: {gnd['respostas_avaliadas']}")
print(f"  Groundedness média: {gnd['groundedness_media']*100:.0f} %")
print(f"  Respostas 100 % fundamentadas: {gnd['respostas_100pct_fundamentadas']*100:.0f} %")
print(f"  Fundamentadas mas doc errado: {gnd['fundamentadas_mas_documento_errado']}")
print(f"  Observação: {gnd['observacao']}")
print()
print("REVISÃO HUMANA DA AMOSTRA")
print(f"  N revisadas: {rev['n']}")
print(f"  Responde à pergunta: {rev['responde_a_pergunta']*100:.0f} %")
print(f"  Fundamentada: {rev['fundamentada']*100:.0f} %")
print(f"  Revisor: {rev['revisor']}")

GROUNDEDNESS (respostas fundamentadas nos trechos recuperados)
  Respostas avaliadas: 43
  Groundedness média: 100 %
  Respostas 100 % fundamentadas: 100 %
  Fundamentadas mas doc errado: 1
  Observação: a resposta atual é extrativa (trecho top-1), por isso é sempre fundamentada; o risco real é responder com o trecho de outro documento

REVISÃO HUMANA DA AMOSTRA
  N revisadas: 13
  Responde à pergunta: 92 %
  Fundamentada: 100 %
  Revisor: sessão de IA (Claude), 02/11/2026 — pendente de confirmação por um membro da equipe


## 11. Versionamento do Modelo e da Base de Conhecimento

Todo score gerado pela API inclui `versao_modelo` e `calculado_em`. Toda resposta do chat inclui `versao_base` e `fonte`.

In [ ]:
art = r["artefatos"]
print("VERSIONAMENTO DO MODELO")
print(f"  Versão:        {mv['versao']}")
print(f"  Treinado em:   {mv['treinado_em']}")
print(f"  SHA256:        {mv['sha256']}")
print(f"  Algoritmo:     {mv['algoritmo']}")
print(f"  Calibração:    {mv['calibracao']}")
print(f"  Limiares ativos:     amarelo={mv['limiares_ativos']['amarelo']}  vermelho={mv['limiares_ativos']['vermelho']}")
print(f"  Limiares recomendados: amarelo={mv['limiares_recomendados']['amarelo']}  vermelho={mv['limiares_recomendados']['vermelho']}")
print(f"  Endpoint:      GET /risco/modelo  (expõe versão, algoritmo, limiares, features)")
print()
print("VERSIONAMENTO DA BASE DE CONHECIMENTO")
print("  Versão por documento: tabela 'documentos.versao_conteudo' no Supabase")
print("  Hash do conjunto: exposto na auditoria de cada resposta do chat (campo 'versao_base')")
print("  Visível em: GET /health (campo 'versao_modelo_risco' para o modelo)")

VERSIONAMENTO DO MODELO
  Versão:        risco-2.0.0
  Treinado em:   2026-11-02T21:42:42+00:00
  SHA256:        c98263417baaec72e18a01e23b6e8c17b5134785e2bb4c68bc0fb291dc7ed66e
  Algoritmo:     XGBoost monotônico
  Calibração:    isotonic
  Limiares ativos:     amarelo=0.3  vermelho=0.6
  Limiares recomendados: amarelo=0.1  vermelho=0.6
  Endpoint:      GET /risco/modelo  (expõe versão, algoritmo, limiares, features)

VERSIONAMENTO DA BASE DE CONHECIMENTO
  Versão por documento: tabela 'documentos.versao_conteudo' no Supabase
  Hash do conjunto: exposto na auditoria de cada resposta do chat (campo 'versao_base')
  Visível em: GET /health (campo 'versao_modelo_risco' para o modelo)


## 12. Endpoint de Inferência Integrado ao Aplicativo

A tela do atendente chama `POST /risco/inferir` com o modelo **final** carregado de `ml/artifacts/risco_v2.joblib`. A resposta inclui score, categoria, fatores, versão do modelo e data.

In [ ]:
import json
exemplo_resposta = {
    "probabilidade": 0.712,
    "categoria": "vermelho",
    "score": {"geral": 0.712, "financeiro": 0.68, "academico": 0.45, "relacionamento": 0.30},
    "fatores": [
        {"codigo": "valor_periodo", "descricao": "Valor do período abaixo da mediana",
         "dimensao": "financeira", "direcao": "aumenta", "peso": 0.42},
        {"codigo": "media_notas", "descricao": "Média de notas 48/100",
         "dimensao": "academica", "direcao": "aumenta", "peso": 0.31},
    ],
    "recomendacao": "Sugestão: contato prioritário. A decisão é do atendente.",
    "versao_modelo": "risco-2.0.0",
    "calculado_em": "2026-11-06T18:00:00Z",
    "persistido": True,
    "score_id": "uuid-do-score",
}
print("Exemplo de resposta do POST /risco/inferir (dados sintéticos):")
print(json.dumps(exemplo_resposta, indent=2, ensure_ascii=False))
print()
print("Modelo carregado: ml/artifacts/risco_v2.joblib (XGBoost, versão risco-2.0.0)")
print("Fallback: placeholder RL treinado nos seeds sintéticos (versão rl-placeholder-seeds-*)")

Exemplo de resposta do POST /risco/inferir (dados sintéticos):
{
  "probabilidade": 0.712,
  "categoria": "vermelho",
  "score": {
    "geral": 0.712,
    "financeiro": 0.68,
    "academico": 0.45,
    "relacionamento": 0.3
  },
  "fatores": [
    {
      "codigo": "valor_periodo",
      "descricao": "Valor do período abaixo da mediana",
      "dimensao": "financeira",
      "direcao": "aumenta",
      "peso": 0.42
    },
    {
      "codigo": "media_notas",
      "descricao": "Média de notas 48/100",
      "dimensao": "academica",
      "direcao": "aumenta",
      "peso": 0.31
    }
  ],
  "recomendacao": "Sugestão: contato prioritário. A decisão é do atendente.",
  "versao_modelo": "risco-2.0.0",
  "calculado_em": "2026-11-06T18:00:00Z",
  "persistido": true,
  "score_id": "uuid-do-score"
}

Modelo carregado: ml/artifacts/risco_v2.joblib (XGBoost, versão risco-2.0.0)
Fallback: placeholder RL treinado nos seeds sintéticos (versão rl-placeholder-seeds-*)


## 13. Resumo Final — Tabela de Itens do Enunciado

Verifica o atendimento de cada item da Entrega 2 com o arquivo de evidência.

In [ ]:
itens = [
    ("Comparação de modelos (LR, RF, XGBoost + CV)",      "PRONTO", "ml/artifacts/metrics.json"),
    ("AUC-ROC, PR-AUC, Recall, F1, Brier, matr. conf.",   "PRONTO", "ml/artifacts/metrics.json + figuras"),
    ("Ajuste de limiar / cobertura 5/10/20 %",             "PRONTO", "ml/artifacts/metrics.json > capacidade"),
    ("Calibração e faixas Verde/Amarelo/Vermelho",         "PRONTO", "ml/artifacts/metrics.json > faixas + risco_calibracao.png"),
    ("Análise de erros (FN/FP, 5 observações)",            "PRONTO", "ml/artifacts/metrics.json > erros"),
    ("Abstenção modelo de risco (outliers)",               "PRONTO", "ml/artifacts/resultados_e2.json > abstencao"),
    ("Equidade por subgrupos (turno, ingresso, etc.)",     "PRONTO", "ml/artifacts/metrics.json > equidade + figuras"),
    ("Variante sem Sexo/Estado Civil",                     "PRONTO", "ml/artifacts/MODEL_VERSION.json + metrics.json"),
    ("Explicabilidade TreeSHAP (3 exemplos anon.)",        "PRONTO", "ml/artifacts/metrics.json > explicabilidade + figuras"),
    ("RAG: 60 perguntas (dir./reform./cont./fora)",        "PRONTO", "ml/rag_eval/perguntas.jsonl"),
    ("RAG: Recall@1/3, MRR, groundedness, abstenção",     "PRONTO", "ml/artifacts/rag_metrics.json"),
    ("RAG: Comparação TF-IDF vs embeddings",               "PRONTO", "ml/artifacts/rag_metrics.json"),
    ("Versionamento modelo (MODEL_VERSION.json)",          "PRONTO", "ml/artifacts/MODEL_VERSION.json"),
    ("Versionamento base conhecimento",                    "PRONTO", "backend/app/api/chat.py + auditoria"),
    ("Endpoint /risco/inferir integrado",                  "PRONTO", "backend/app/api/risco.py + tests/test_risco.py"),
    ("Notebook final executável sem erros",                "PRONTO", "ml/Entrega2_IA_Final.ipynb (este)"),
    ("Script consolidar_resultados.py",                    "PRONTO", "ml/consolidar_resultados.py"),
    ("Relatório de resultados",                            "PRONTO", "docs/entrega_ia/relatorio_resultados.md"),
    ("Model Card v2.0",                                    "PRONTO", "docs/entrega_ia/model_card_v2.md"),
]
print(f"{'Item':<52} {'Status':>8}  Evidência")
print("-" * 100)
for item, status, evidencia in itens:
    print(f"{item:<52} {status:>8}  {evidencia}")
print()
todos_prontos = all(s == "PRONTO" for _, s, _ in itens)
print(f"Todos os itens: {'✓ COMPLETO' if todos_prontos else '⚠ INCOMPLETO'}")

Item                                                   Status  Evidência
----------------------------------------------------------------------------------------------------
Comparação de modelos (LR, RF, XGBoost + CV)           PRONTO  ml/artifacts/metrics.json
AUC-ROC, PR-AUC, Recall, F1, Brier, matr. conf.        PRONTO  ml/artifacts/metrics.json + figuras
Ajuste de limiar / cobertura 5/10/20 %                 PRONTO  ml/artifacts/metrics.json > capacidade
Calibração e faixas Verde/Amarelo/Vermelho             PRONTO  ml/artifacts/metrics.json > faixas + risco_calibracao.png
Análise de erros (FN/FP, 5 observações)                PRONTO  ml/artifacts/metrics.json > erros
Abstenção modelo de risco (outliers)                   PRONTO  ml/artifacts/resultados_e2.json > abstencao
Equidade por subgrupos (turno, ingresso, etc.)         PRONTO  ml/artifacts/metrics.json > equidade + figuras
Variante sem Sexo/Estado Civil                         PRONTO  ml/artifacts/MODEL_VERSION.json + metr